# AI4I condition classifier: local / Workbench

Educational synthetic data, no future failure horizon. This notebook reuses the same modules, split seed and selection procedure as the local application. No cloud resources or BigQuery jobs are created by these cells. Use Python 3.11. Install requirements-dev.txt in that kernel before running. In Workbench, upload the project without credentials or .venv and select the project root below.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidates if (p / 'src' / 'train.py').exists()), None)
if project_root is None:
    raise RuntimeError('Open this notebook inside the uploaded project/cloud directory.')
os.chdir(project_root)
sys.path.insert(0, str(project_root))
print('Project:', project_root, '\nPython:', sys.version)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError('Use a Python 3.11 kernel for the locked experiment.')

In [ ]:
from src.data_loader import download_dataset, load_dataset, dataset_report
from src.config import DATA_PATH
if not DATA_PATH.exists():
    download_dataset()
dataset = load_dataset()
report = dataset_report(dataset)
print({key: report[key] for key in ['rows', 'target_counts', 'label_inconsistencies']})
dataset.head()

## Preprocessing, fitting and validation-only selection

The same module creates a 60/20/20 stratified split, scaled/OHE LR and OHE RF, three-fold training-internal calibration candidates, validation-selected thresholds and the frozen model selection before test evaluation. Existing artifacts are never silently retrained. For a new experiment, archive models/ and artifacts/ before starting.

In [ ]:
from src.train import split_dataset
from src.preprocessing import model_candidates
splits = split_dataset(dataset)
print({name: {'rows': len(part), 'failures': int(part['Machine failure'].sum())} for name, part in splits.items()})
model_candidates()

In [ ]:
import subprocess
from src.config import MODELS
if not (MODELS / 'metadata.json').exists():
    subprocess.run([sys.executable, '-m', 'src.train'], check=True)
else:
    print('Loading the completed original experiment; no retraining performed.')
if not (MODELS / 'probability_metadata.json').exists():
    subprocess.run([sys.executable, '-m', 'src.probability_profile'], check=True)
else:
    print('Loading the persisted calibrated probability profile.')

In [ ]:
import json
import pandas as pd
from src.config import ARTIFACTS
results = json.loads((ARTIFACTS / 'evaluation.json').read_text())
metrics = ['accuracy', 'precision', 'recall', 'f1', 'average_precision', 'roc_auc', 'brier_score']
comparison = pd.DataFrame({name: {key: result['test'][key] for key in metrics} for name, result in results.items()}).T
comparison

In [ ]:
from src.inference import Predictor
from src.config import FEATURES
predictor = Predictor.load()
print(predictor.metadata['selected_model'], predictor.metadata['threshold'], predictor.metadata['score_kind'])
print('Classification policies:', predictor.metadata['policies'])
predictor.predict(dataset[FEATURES].head())

In [ ]:
subprocess.run([sys.executable, 'scripts/export_reports.py'], check=True)
print('Exported artifacts/test_metrics.csv, artifacts/figures/, docs/evaluation_report.md')
print('Model, schema and metadata:', MODELS)

## Interpretation and cleanup

Test results belong to the frozen local methodology, not the reference experiment. The original winner is uncalibrated RF; the dashboard uses the separate calibrated RF probability profile. Both policy thresholds use validation evidence, and profile metrics reuse the previously inspected test partition. Small failure counts, synthetic data and missing external validation limit interpretation. Cloud execution remains unverified until actually run in Workbench. Close/delete any paid notebook resources and inspect retained disks after the assessment.